In [73]:
import pandas as pd

In [74]:
data = pd.read_csv("../data/FraudDetectionDataset.csv")
data.head()

,Transaction_ID,User_ID,Transaction_Amount,Transaction_Type,Time_of_Transaction,Device_Used,Location,Previous_Fraudulent_Transactions,Account_Age,Number_of_Transactions_Last_24H,Payment_Method,Fraudulent
0,T1,4174,1292.76,ATM Withdrawal,16.0,Tablet,San Francisco,0,119,13,Debit Card,0
1,T2,4507,1554.58,ATM Withdrawal,13.0,Mobile,New York,4,79,3,Credit Card,0
2,T3,1860,2395.02,ATM Withdrawal,NaN,Mobile,NaN,3,115,9,NaN,0
3,T4,2294,100.10,Bill Payment,15.0,Desktop,Chicago,4,3,4,UPI,0
4,T5,2130,1490.50,POS Payment,19.0,Mobile,San Francisco,2,57,7,Credit Card,0


In [75]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 51000 entries, 0 to 50999
Data columns (total 12 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   Transaction_ID                    51000 non-null  str    
 1   User_ID                           51000 non-null  int64  
 2   Transaction_Amount                48480 non-null  float64
 3   Transaction_Type                  51000 non-null  str    
 4   Time_of_Transaction               48448 non-null  float64
 5   Device_Used                       48527 non-null  str    
 6   Location                          48453 non-null  str    
 7   Previous_Fraudulent_Transactions  51000 non-null  int64  
 8   Account_Age                       51000 non-null  int64  
 9   Number_of_Transactions_Last_24H   51000 non-null  int64  
 10  Payment_Method                    48531 non-null  str    
 11  Fraudulent                        51000 non-null  int64  
dtypes: float64(2), 

In [76]:
x = data.drop(columns=["Transaction_ID","User_ID","Fraudulent"])
y = data["Fraudulent"]
print(x)
print(y)

       Transaction_Amount Transaction_Type  Time_of_Transaction Device_Used  \
0                 1292.76   ATM Withdrawal                 16.0      Tablet   
1                 1554.58   ATM Withdrawal                 13.0      Mobile   
2                 2395.02   ATM Withdrawal                  NaN      Mobile   
3                  100.10     Bill Payment                 15.0     Desktop   
4                 1490.50      POS Payment                 19.0      Mobile   
...                   ...              ...                  ...         ...   
50995             3112.51     Bill Payment                 15.0      Mobile   
50996             2897.15  Online Purchase                  3.0      Mobile   
50997             2204.43      POS Payment                 18.0      Mobile   
50998             4787.17      POS Payment                 19.0      Tablet   
50999              814.72      POS Payment                  3.0      Tablet   

            Location  Previous_Fraudulent_Transacti

In [77]:
from sklearn.model_selection import train_test_split

x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=42)

print(x_train.shape)
print(x_test.shape)
print(y_train.shape)
print(y_test.shape)

(40800, 9)
(10200, 9)
(40800,)
(10200,)


In [78]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer


num_features = x.select_dtypes(include=["int64","float64"]).columns
cat_features = x.select_dtypes(include=["object"]).columns

num_pipeline = Pipeline([
    ("imputer",SimpleImputer(strategy="mean")),
    ("scaler",StandardScaler())
])

cat_pipeline = Pipeline([
    ("imputer",SimpleImputer(strategy="most_frequent")),
    ("encoder",OneHotEncoder(handle_unknown="ignore",sparse_output=False))
])

preprocessor = ColumnTransformer([
    ("num",num_pipeline,num_features),
    ("cat",cat_pipeline,cat_features)
])

C:\Users\nigam\AppData\Local\Temp\ipykernel_12852\3313224236.py:8: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_features = x.select_dtypes(include=["object"]).columns


In [79]:
x_train_preprocessed = preprocessor.fit_transform(x_train)
x_test_preprocessed = preprocessor.transform(x_test)
print(x_train_preprocessed.shape)
print(x_test_preprocessed.shape)

(40800, 27)
(10200, 27)


In [80]:
import torch
x_train = torch.tensor(x_train_preprocessed, dtype=torch.float32)
y_train = torch.tensor(y_train.values, dtype=torch.float32).view(-1,1)

x_test = torch.tensor(x_test_preprocessed, dtype=torch.float32)
y_test = torch.tensor(y_test.values, dtype=torch.float32).view(-1,1)

In [81]:
from torch.utils.data import TensorDataset,DataLoader

train_dataset = TensorDataset(x_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)



In [82]:
import torch.nn as nn

class MLP(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(input_size, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.model(x)

In [83]:
model = MLP(input_size=x_train.shape[1])

criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(),lr=0.0005)

In [84]:
epochs = 1001

for epoch in range(epochs):
    model.train()
    total_loss = 0
    for xb,yb in train_loader:
        output = model(xb)
        loss = criterion(output,yb)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        avg_loss = total_loss / len(train_loader)
        
    if epoch % 10 == 0:
        print(f"Epoch {epoch}, Loss: {total_loss:.4f} , avg_loss: {avg_loss:.4f}")

Epoch 0, Loss: 266.2366 , avg_loss: 0.2088
Epoch 10, Loss: 244.1424 , avg_loss: 0.1915
Epoch 20, Loss: 234.7085 , avg_loss: 0.1841
Epoch 30, Loss: 223.7784 , avg_loss: 0.1755
Epoch 40, Loss: 213.6243 , avg_loss: 0.1675
Epoch 50, Loss: 203.6276 , avg_loss: 0.1597
Epoch 60, Loss: 194.7773 , avg_loss: 0.1528
Epoch 70, Loss: 186.7345 , avg_loss: 0.1465
Epoch 80, Loss: 178.9856 , avg_loss: 0.1404
Epoch 90, Loss: 173.6889 , avg_loss: 0.1362
Epoch 100, Loss: 167.9959 , avg_loss: 0.1318
Epoch 110, Loss: 162.1064 , avg_loss: 0.1271
Epoch 120, Loss: 156.6614 , avg_loss: 0.1229
Epoch 130, Loss: 152.8714 , avg_loss: 0.1199
Epoch 140, Loss: 146.8855 , avg_loss: 0.1152
Epoch 150, Loss: 144.0233 , avg_loss: 0.1130
Epoch 160, Loss: 140.6956 , avg_loss: 0.1103
Epoch 170, Loss: 136.4745 , avg_loss: 0.1070
Epoch 180, Loss: 132.4608 , avg_loss: 0.1039
Epoch 190, Loss: 128.7899 , avg_loss: 0.1010
Epoch 200, Loss: 125.1013 , avg_loss: 0.0981
Epoch 210, Loss: 120.2806 , avg_loss: 0.0943
Epoch 220, Loss: 117.

In [ ]:
with torch.no_grad():
    preds = model(x_test)
    predicted = (preds > 0.5).float()
    
print(predicted[:10])    

accuracy = (predicted == y_test).float().mean()
print("Accuracy:", accuracy.item())

tensor([[0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [1.],
        [0.],
        [0.]])
Accuracy: 0.9100980162620544
